# Lekcija 05 - Agentni RAG


## Namestitev

Ta zvezek prikazuje vzorec Agentic RAG (Retrieval-Augmented Generation) z uporabo Microsoft Agent Framework.

**Pogoji:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — konec vaše storitve Azure AI Search
- `AZURE_SEARCH_API_KEY` — vaš ključ API za Azure AI Search
- Azure OpenAI namestitev nastavljena preko spremenljivk okolja
- Azure CLI avtentikacija opravljena (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Kaj je Agentic RAG?

Tradicionalni RAG sledi fiksnemu postopku: najprej iskanje dokumentov, nato generiranje odgovora. **Agentic RAG** gre korak dlje tako, da agentu daje avtonomijo, da odloči **kdaj** in **kako** pridobivati informacije.

Z Agentic RAG lahko agent:
- **Odloča**, ali je pred odgovorom na vprašanje potreben vnos podatkov
- **Izbere**, kateri vir podatkov ali orodje bo uporabljal za poizvedbo
- **Ocenjuje** pridobljene rezultate in izvede dodatna iskanja, če prvi poskus ni zadosten
- **Združuje** informacije iz več korakov iskanja v koherenten odgovor

To agentu omogoča večjo prilagodljivost in natančnost v primerjavi s statičnim postopkom iskanja in nato generiranja odgovora.


## Ustvarjanje iskalnega orodja

V Agentic RAG so zunanji viri podatkov oviti kot **orodja**, ki jih agent lahko po potrebi pokliče. To agentu omogoča, da obravnava iskanje kot še eno dejanje, ki ga lahko izvede, namesto kot obvezni korak.

Spodaj definiramo bazo znanja o potovanjih in jo predstavimo kot orodje, ki ga agent lahko uporabi za iskanje informacij o destinacijah.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Izdelava RAG agenta

Zdaj ustvarimo agenta, ki mu je naročeno, da **vedno pridobi informacije pred odgovorom**. Agent uporablja orodje `search_travel_knowledge`, da osredotoči svoje odgovore na bazo znanja namesto, da bi se zanašal na lastne podatke za usposabljanje.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iterativno iskanje — vzorec Maker-Checker

Ključna prednost Agentic RAG je **iterativno iskanje**. Agent lahko izvede več krogov iskanja, da preveri, izboljša ali razširi svoje začetne ugotovitve — podobno kot delovni tok "maker-checker":

1. **Korak maker**: Agent pridobi začetne informacije in pripravi osnutek odgovora.
2. **Korak checker**: Agent izvede dodatna iskanja za preverjanje podrobnosti ali zapolnitev vrzeli.

Spodaj agentu zastavijo vprašanje, ki zahteva primerjavo več destinacij, kar ga spodbuja k večkratnemu iskanju.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Povzetek

V tej lekciji ste se naučili, kako zgraditi sistem **Agentic RAG** z uporabo Microsoft Agent Framework:

- **Agentic RAG** agentom omogoča, da samostojno odločajo, kdaj pridobiti informacije, s čimer je pridobivanje informacij dinamično in ne fiksno.
- **Orodja kot viri podatkov**: Zunanje zbirke znanja (kot je Azure AI Search) so zavite kot orodja, ki jih lahko agent kliče.
- **Iterativno pridobivanje**: Vzorec maker-checker omogoča agentu, da izvede več krogov pridobivanja — iskanje, preverjanje in izpopolnjevanje — preden poda končni odgovor.

V produkciji bi zamenjali v pomnilniku shranjeno bazo `TRAVEL_KNOWLEDGE_BASE` z resničnim indeksom Azure AI Search, ki lahko obvladuje velik obseg pridobivanja potovalnih dokumentov.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
